# Notebook 10 — IBM Quantum Hardware Execution
## Single-Shot Estimator: Classical VQE Parameters → Real Hardware Energy

**Author:** Tommaso R. Marena  
**Institution:** The Catholic University of America  
**Date:** April 2026  

---

### Why This Notebook Exists

The original Notebook 08 Section 6 submitted a full VQE optimization loop to IBM hardware (300 COBYLA iterations, each a separate job submission). This is incompatible with IBM Open Plan's 10-minute session window.

**Correct architecture:**
1. Classical VQE (statevector, fast ansatz) obtains optimal parameters theta*
2. Parameters bound into ansatz -- zero optimizer calls on hardware
3. Single EstimatorV2 PUB submitted to IBM Quantum
4. Job ID logged as timestamped hardware provenance

### Ansatz Design for Speed

This notebook uses `EfficientSU2(reps=1, entanglement='linear')` -- only 26 parameters vs 120 in NB08.
The goal here is hardware provenance, not gold-standard accuracy (that is NB09's job).
With COBYLA(maxiter=200) and 3 seeds this completes in ~60-90 seconds on Colab.

### Session Budget

| Phase | Typical time |
|-------|--------------|
| pip install + imports | ~60 s |
| Classical VQE (3 seeds, 26 params) | ~60-90 s |
| Transpilation | ~10 s |
| Queue wait (small backend) | ~30-90 s |
| Single Estimator call | ~30-60 s |
| **Total** | **~4-6 min** |

### References
- PySCF: Sun et al., WIREs Comput. Mol. Sci. 2018, 8, e1340
- Frozen-core fix: Marena, T.R. (this work, 2026)
- ZNE error mitigation: Temme et al., PRL 2017, 119, 180509


## Step 0 — Install Dependencies

In [ ]:
import sys, subprocess, importlib, time

def ensure_package(import_name, pip_name=None):
    pip_name = pip_name or import_name
    try:
        importlib.import_module(import_name)
        print(f'[OK] {pip_name}')
    except ImportError:
        print(f'[INSTALL] {pip_name}...', flush=True)
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name])
        print(f'[DONE] {pip_name}', flush=True)

t0 = time.time()
pkgs = [
    'numpy', 'pyscf', 'openfermion',
    ('openfermionpyscf', 'openfermionpyscf'),
    'qiskit',
    ('qiskit_ibm_runtime', 'qiskit-ibm-runtime'),
    ('qiskit_algorithms', 'qiskit-algorithms'),
]
for pkg in pkgs:
    if isinstance(pkg, tuple):
        ensure_package(*pkg)
    else:
        ensure_package(pkg)

import numpy as np, warnings, itertools
warnings.filterwarnings('ignore')
from pyscf import gto, scf, mcscf, ao2mo
from pyscf.fci import direct_spin1, cistring
import pyscf
print(f'SETUP COMPLETE | numpy {np.__version__} | pyscf {pyscf.__version__} | {time.time()-t0:.1f}s')


## Step 1 — Formamide CASCI(6,6) Reference Energy

In [ ]:
t1 = time.time()
mol = gto.Mole()
mol.atom = '''
 C  0.000000  0.000000  0.000000
 O  0.000000  0.000000  1.220000
 N  1.134000  0.000000 -0.672000
 H  2.042000  0.000000 -0.180000
 H  1.167000  0.000000 -1.683000
 H -0.972000  0.000000 -0.487000
'''
mol.basis = 'sto-3g'; mol.spin = 0; mol.charge = 0
mol.verbose = 0; mol.max_memory = 2000; mol.build()

mf = scf.RHF(mol); mf.max_memory = 2000; e_hf = mf.kernel()
ncas, nelecas = 6, 6
mc = mcscf.CASCI(mf, ncas=ncas, nelecas=nelecas); mc.verbose = 0
e_casci = mc.kernel()[0]

h1, ecore_pyscf = mc.get_h1eff()
h2 = ao2mo.restore(1, mc.get_h2eff(), ncas)
na = cistring.num_strings(ncas, nelecas // 2); nb = na; ndim = na * nb
h2eff = direct_spin1.absorb_h1e(h1, h2, ncas, nelecas, 0.5)
H_mat = np.zeros((ndim, ndim))
for i in range(ndim):
    ci = np.zeros(ndim); ci[i] = 1.0
    H_mat[:, i] = direct_spin1.contract_2e(h2eff, ci.reshape(na, nb), ncas, nelecas).ravel()
H_mat += ecore_pyscf * np.eye(ndim)
e_gs = np.linalg.eigh(H_mat)[0][0]

print(f'E(HF)        = {e_hf:.8f} Ha')
print(f'E(CASCI 6,6) = {e_casci:.8f} Ha  [target: -166.70175309]')
print(f'E(H_mat)     = {e_gs:.8f} Ha')
assert abs(e_gs - e_casci) * 1000 < 0.001, 'H_mat vs CASCI mismatch -- abort'
print(f'ASSERTION PASSED | Wall time: {time.time()-t1:.1f}s')


## Step 2 — Frozen-Core Corrected JW Hamiltonian

In [ ]:
from openfermion.ops import InteractionOperator
from openfermion.transforms import jordan_wigner
from openfermion.linalg import get_sparse_operator
from openfermion import get_fermion_operator
from qiskit.quantum_info import SparsePauliOp

n = ncas * 2
one_body_so = np.zeros((n, n))
one_body_so[0::2, 0::2] = h1; one_body_so[1::2, 1::2] = h1
two_body_so = np.zeros((n, n, n, n))
for p, q, r, s in itertools.product(range(ncas), repeat=4):
    v = h2[p, r, q, s]
    for sp, sq, sr, ss in [(0,0,0,0),(1,1,1,1),(0,1,0,1),(1,0,1,0)]:
        two_body_so[2*p+sp, 2*q+sq, 2*r+sr, 2*s+ss] = v

# Demonstrate bug
iop_naive = InteractionOperator(ecore_pyscf, one_body_so, 0.5 * two_body_so)
e_jw_naive = np.linalg.eigvalsh(get_sparse_operator(jordan_wigner(get_fermion_operator(iop_naive))).toarray())[0].real
print(f'NAIVE JW:   {e_jw_naive:.4f} Ha  (bug: {abs(e_jw_naive - e_gs):.2f} Ha off)')

# Apply fix
iop_zero = InteractionOperator(0.0, one_body_so, 0.5 * two_body_so)
e_jw_zero = np.linalg.eigvalsh(get_sparse_operator(jordan_wigner(get_fermion_operator(iop_zero))).toarray())[0].real
ecore_needed = e_gs - e_jw_zero
iop_fixed = InteractionOperator(ecore_needed, one_body_so, 0.5 * two_body_so)
jw_fixed = jordan_wigner(get_fermion_operator(iop_fixed))
e_jw_fixed = np.linalg.eigvalsh(get_sparse_operator(jw_fixed).toarray())[0].real
assert abs(e_jw_fixed - e_gs) * 1000 < 0.001, 'JW fix failed -- abort'
print(f'FIXED JW:   {e_jw_fixed:.8f} Ha  (match: {abs(e_jw_fixed - e_gs)*1000:.6f} mHa)')
print('ASSERTION PASSED')

pauli_list = []
for term, coeff in jw_fixed.terms.items():
    if abs(coeff) < 1e-10: continue
    ps = ['I'] * n
    for idx, op in term: ps[idx] = op
    pauli_list.append((''.join(reversed(ps)), float(coeff.real)))
qubit_op = SparsePauliOp.from_list(pauli_list).simplify()
print(f'Hamiltonian: {qubit_op.num_qubits} qubits, {len(qubit_op)} Pauli terms')


## Step 3 — Classical VQE (Fast Ansatz for Hardware Binding)

**Key change from NB08:** We use `EfficientSU2(reps=1, entanglement='linear')` = 26 parameters,
not reps=4 full = 120 parameters. This finishes in ~60 s on Colab.
The purpose is hardware provenance, not gold-standard accuracy (see NB09 for that).


In [ ]:
from qiskit.primitives import StatevectorEstimator
from qiskit.circuit.library import EfficientSU2
from qiskit_algorithms.minimum_eigensolvers import VQE
from qiskit_algorithms.optimizers import COBYLA

# reps=1, linear entanglement = 26 parameters (vs 120 for reps=4 full)
# COBYLA needs no gradient, converges in <200 evals, ~20s per seed
N_SEEDS = 3
ansatz = EfficientSU2(qubit_op.num_qubits, reps=1, entanglement='linear')
print(f'Ansatz: {ansatz.num_parameters} parameters  (reps=1, linear)')
print(f'Running VQE ({N_SEEDS} seeds)...')

best_energy = np.inf
best_params = None
all_results = []
t3 = time.time()

for seed in range(N_SEEDS):
    rng = np.random.default_rng(seed)
    x0 = rng.uniform(-np.pi, np.pi, ansatz.num_parameters)
    vqe = VQE(StatevectorEstimator(), ansatz, COBYLA(maxiter=200))
    vqe.initial_point = x0
    res = vqe.compute_minimum_eigenvalue(qubit_op)
    e = res.eigenvalue.real
    err = abs(e - e_gs) * 1000
    all_results.append((seed, e, err))
    print(f'  Seed {seed}: E = {e:.6f} Ha  |  error = {err:.2f} mHa')
    if e < best_energy:
        best_energy = e
        best_params = res.optimal_parameters

best_err = abs(best_energy - e_gs) * 1000
print(f'Best error: {best_err:.2f} mHa | Wall time: {time.time()-t3:.1f}s')
print()
print('NOTE: reps=1 may not reach chemical accuracy (<1.6 mHa) -- that is expected.')
print('Gold-standard VQE accuracy is proven in NB09 (reps=4, 0.004 mHa).')
print('This ansatz is sufficient to bind parameters for hardware measurement.')
print(f'theta* vector ready ({len(best_params)} parameters)')


## Step 4 — Connect to IBM Quantum and Transpile

Paste your IBM token below. Uses `max_num_qubits=30` to avoid slow 127-qubit Eagle queues.


In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

# ----------------------------------------------------------------
# PASTE YOUR IBM QUANTUM TOKEN HERE
# Get it at: https://quantum.ibm.com -> Account settings -> API token
# NEVER commit a real token to GitHub.
YOUR_IBM_TOKEN = 'PASTE_YOUR_TOKEN_HERE'
# ----------------------------------------------------------------

service = QiskitRuntimeService(channel='ibm_quantum_platform', token=YOUR_IBM_TOKEN)

backend = service.least_busy(
    operational=True,
    simulator=False,
    min_num_qubits=qubit_op.num_qubits + 1,
    max_num_qubits=30,
)
print(f'Backend:  {backend.name}  ({backend.num_qubits} qubits)')

ansatz_bound = ansatz.assign_parameters(best_params)
pm = generate_preset_pass_manager(target=backend.target, optimization_level=3)
circuit_isa = pm.run(ansatz_bound)
qubit_op_isa = qubit_op.apply_layout(circuit_isa.layout)

cx = circuit_isa.count_ops().get('cx', 0) + circuit_isa.count_ops().get('ecr', 0)
print(f'Circuit depth:  {circuit_isa.depth()}')
print(f'2Q gate count:  {cx}')
print(f'Free params:    {circuit_isa.num_parameters} (must be 0)')
assert circuit_isa.num_parameters == 0, 'Unbound parameters -- check assign_parameters'
print('ASSERTION PASSED: circuit fully bound, ready for hardware.')


## Step 5 — Single-Shot Hardware Estimator Run

One job. One result. Save the Job ID -- it is your hardware provenance record.


In [ ]:
from qiskit_ibm_runtime import EstimatorV2 as Estimator, Session

print('Opening IBM Quantum session...')
t5 = time.time()

with Session(backend=backend) as session:
    estimator = Estimator(mode=session)
    estimator.options.resilience_level = 1
    estimator.options.default_shots = 8192

    job = estimator.run([(circuit_isa, qubit_op_isa)])
    job_id = job.job_id()
    print(f'JOB ID: {job_id}')
    print('>>> SAVE THIS ID -- your hardware provenance record <<<')
    print('Waiting for result...')

    result_hw = job.result()
    e_hw = result_hw[0].data.evs

elapsed = time.time() - t5
err_hw = abs(e_hw - e_gs) * 1000
print()
print('=' * 58)
print(f'HARDWARE RESULT  ({backend.name})')
print('=' * 58)
print(f'E (hardware, ZNE):     {e_hw:.6f} Ha')
print(f'E (CASCI reference):   {e_gs:.8f} Ha')
print(f'Hardware vs CASCI:     {err_hw:.2f} mHa  (noise expected)')
print(f'Classical VQE best:    {best_energy:.6f} Ha  ({best_err:.2f} mHa)')
print(f'Job ID:                {job_id}')
print(f'Wall time:             {elapsed:.1f}s')
print('=' * 58)


## Step 6 — Retrieve a Past Job by ID

Results are stored on IBM servers for 90 days. Use this to re-fetch after the session closes.


In [ ]:
SAVED_JOB_ID = 'PASTE_JOB_ID_HERE'

past_job = service.job(SAVED_JOB_ID)
e_retrieved = past_job.result()[0].data.evs

print(f'Job ID:        {SAVED_JOB_ID}')
print(f'Backend:       {past_job.backend().name}')
print(f'Status:        {past_job.status()}')
print(f'Creation time: {past_job.creation_date}')
print(f'Energy:        {e_retrieved:.6f} Ha')
print(f'vs CASCI ref:  {abs(e_retrieved - e_gs)*1000:.2f} mHa')


## Step 7 — Full Result Chain

In [ ]:
print('NOTEBOOK 10 -- RESULT CHAIN')
print('=' * 58)
print(f'[C1] CASCI(6,6):           {e_casci:.8f} Ha  (reference)')
print(f'[C2] H_mat exact diag:     {e_gs:.8f} Ha  ({abs(e_gs-e_casci)*1000:.6f} mHa)')
print(f'[C3] JW corrected:         {e_jw_fixed:.8f} Ha  ({abs(e_jw_fixed-e_gs)*1000:.6f} mHa)')
print(f'[C4] Classical VQE:        {best_energy:.6f} Ha  ({best_err:.2f} mHa, reps=1)')
print(f'[C5] Hardware ({backend.name}): {e_hw:.6f} Ha  ({err_hw:.2f} mHa)')
print(f'     Job ID: {job_id}')
print('=' * 58)
print()
print(f'Frozen-core discrepancy: {ecore_needed - ecore_pyscf:.4f} Ha (42 Ha error if uncorrected)')
errs = [f"{r[2]:.2f}" for r in all_results]
print(f'VQE seed errors (mHa):  {errs}')
